# Figure 4 — secondary and safety outcomes

**What else happened.** Three secondary outcomes, corrected within their family, and
four safety outcomes that rest on few events and are therefore **descriptive**: estimation, not
testing `[§10, §13]`. The distinction is drawn in the figure itself, because a forest plot that
renders them identically invites reading a safety interval as a test.

This notebook **composes**; it computes nothing. Every number comes off the cached run that
`manuscript.load()` returns, every sentence beneath the exhibit comes off `report.py`, and the
artefacts land in `out/manuscript/`. Section references `[§n]` point to `../statistical_analysis_plan.md`.

## 1. The run

One cached build of Stages 1-11, shared by all seven exhibits. The record below says what it depends on.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Independent of where the notebook was started: find the directory the shipped modules live in.
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path[:0] = [str(_root), str(_root / "figures_and_tables")]

import config as C
import report
import manuscript as M

M.use_house_style()
ARM = C.TREATMENT_LABELS          # {0: "EVT alone", 1: "bridging"}
ARMS = sorted(ARM)                # control first

bundle = M.load()                 # the cached Stages 1-11 run; built once, then read in seconds
bundle.provenance

## 2. The rows

`report.binary_rows` is the builder behind `T10` and `T11`. Taking the rows from it — rather than
assembling them here — is what keeps this figure, Table S2 and the supplement carrying one set of
numbers and one set of statements.

In [ ]:
def frame(family):
    rows, clauses = report.binary_rows(bundle, family)
    return pd.DataFrame(list(rows[1:]), columns=list(rows[0])), clauses


secondary, secondary_clauses = frame("secondary")
safety, safety_clauses = frame("safety")
secondary.iloc[:, :8]

### From the estimates, not from the rendered rows

The table cells are formatted strings. The figure plots the numbers themselves, off the same
estimates and the same intervals.

In [ ]:
def rows_for(family, mark):
    """(label, point, lo, hi, mark) per outcome — the unaugmented weighted risk difference.

    The model-assisted estimates are NOT plotted. They are a second estimate of the same quantity and
    doubling the rows halves the space each interval gets to say what it says; Table S2 carries them,
    beside the unaugmented row they are read with `[§8]`.
    """
    out = []
    for est in bundle.secondary.by_family()[family]:
        interval = bundle.boot.intervals.get(f"{est.outcome}.rd")
        if interval is None:
            continue
        out.append((C.OUTCOMES[est.outcome].label, est.rd, interval.lo, interval.hi, mark))
    return out


rows = rows_for("secondary", M.Mark.ESTIMATE)
boundary = len(rows)
rows += rows_for("safety", M.Mark.DESCRIPTIVE)
len(rows), boundary

## 3. The figure

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 0.34 * len(rows) + 1.4))
M.forest(ax, rows, f"risk difference  ({ARM[1]} − {ARM[0]})", 0.0)

# The two families are separated, because only one of them was tested. The labels sit outside the
# panel on the right, where they cannot collide with an outcome name or with the legend line.
ax.axhline(boundary - 0.5, color="#c8c8c8", linewidth=0.8)
right = ax.get_xlim()[1]
for label, first, last in (("secondary", 0, boundary - 1),
                           ("safety", boundary, len(rows) - 1)):
    ax.text(right, (first + last) / 2, label, rotation=90, va="center", ha="left", fontsize=7,
            style="italic")

# No marker key on the canvas: what the three marks mean is the legend's job, composed from `Mark`.
fig.tight_layout()

## 4. Write it

In [ ]:
caption = (
    "Figure 4. Secondary and safety outcomes: overlap-weighted risk differences with 95% percentile "
    f"intervals. {M.Mark.ESTIMATE.key.capitalize()} are the secondary-outcome estimates and "
    f"{M.Mark.DESCRIPTIVE.key} the safety estimates, whose mark is their label: the safety outcomes "
    f"are {report.DESCRIPTIVE_SAFETY}. The model-assisted augmented estimates are in Table S2.")

# The augmented sentence describes a quantity this figure no longer plots, so it is not carried onto
# it; Table S2 shows those estimates and states it there. Everything else the families require stays.
clauses = [c for c in (*secondary_clauses, *safety_clauses) if "augmented" not in c]

source = pd.DataFrame([(label.strip(), point, lo, hi, mark.name)
                       for label, point, lo, hi, mark in rows],
                      columns=["row", "risk difference", "ci lo", "ci hi", "mark"])

path = M.save_figure("Figure_4", fig, source,
                     (caption, *clauses, M.provenance_clause(bundle)))
M.save_legends()      # rebuilds out/manuscript/figure_legends.md
M.show(path)

## 5. Reading it

- **The families were corrected separately and the primary was not corrected at all** `[§13]`. The
  caption says which correction ran over how many hypotheses, because a forest plot cannot.
- **Model-assisted is not doubly robust** `[§8]`. The augmented point borrows a nuisance model where
  the minority cell supports one; it is read beside the unaugmented estimate, never instead of it.
- **A wide safety interval is not evidence of safety.** These outcomes rest on few events, which is
  what "descriptive" records `[§10, §13]`.